# 🔗 Workflow Integration - Complete SAR Processing System

Welcome to Phase 4 of the Financial Services Agentic AI Project!

In this notebook, you'll integrate both AI agents into a complete **end-to-end SAR processing workflow** that demonstrates real-world financial compliance automation.

## 🎯 Learning Objectives
- Build a complete two-stage AI workflow with human oversight
- Implement human-in-the-loop decision gates for compliance
- Generate complete SAR documents from AI analysis
- Create comprehensive audit trails for regulatory examination
- Demonstrate cost optimization through intelligent agent coordination

## 📋 Business Context
This workflow simulates how banks actually process suspicious activity reports:
1. **Risk Screening**: AI agents analyze transaction patterns for suspicious activity
2. **Human Review**: Compliance officers review AI findings before proceeding
3. **Narrative Generation**: Only approved cases get full compliance documentation
4. **SAR Filing**: Complete regulatory forms are generated for submission
5. **Audit Documentation**: Every decision is logged for regulatory examination

## 🏗️ System Architecture

```
📊 CSV Data → 🔍 Risk Analyst → 👤 Human Decision → ✅ Compliance Officer → 📄 SAR Document
              (Chain-of-Thought)    (Gate)         (ReACT Framework)     (FinCEN Ready)
```

## 🚀 Prerequisites Check

Before starting, ensure you have completed:
- ✅ Phase 1: Foundation components (`foundation_sar.py`)
- ✅ Phase 2: Risk Analyst Agent (`risk_analyst_agent.py`)
- ✅ Phase 3: Compliance Officer Agent (`compliance_officer_agent.py`)
- ✅ Both agents pass their comprehensive test scenarios

If any component is missing, return to previous notebooks to complete implementation.

In [1]:
# Setup and Environment Configuration
import os
import sys
import json
import pandas as pd
import uuid
import hashlib
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('../src'))

# Load environment variables
load_dotenv('../.env')

print("📚 Libraries imported successfully!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path")

📚 Libraries imported successfully!
🔐 Environment variables loaded
📂 Source directory added to Python path


In [2]:
# OpenAI Setup for Vocareum
import openai

# Initialize OpenAI client for Vocareum
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your Vocareum OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        base_url="https://api.openai.com/v1",
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized with Vocareum routing")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    print("📍 Base URL: https://api.openai.com/v1")

✅ OpenAI client initialized with Vocareum routing
🔑 API key: sk-proj-...rGEA
📍 Base URL: https://api.openai.com/v1


In [3]:

from foundation_sar import (
     CustomerData,
     AccountData,
     TransactionData,
     CaseData,
     RiskAnalystOutput,
     ComplianceOfficerOutput,
     ExplainabilityLogger,
     DataLoader,
     load_csv_data
 )
from risk_analyst_agent import RiskAnalystAgent
from compliance_officer_agent import ComplianceOfficerAgent


explainability_logger = ExplainabilityLogger("../outputs/audit_logs/workflow_integration.jsonl")
risk_agent = RiskAnalystAgent(client, explainability_logger)
compliance_agent = ComplianceOfficerAgent(client, explainability_logger)

print("✅ Ready to import components after implementation")

✅ Ready to import components after implementation


In [ ]:
# Quick token capture test — run one case and inspect last_usage
_test_customer = selected_customers[0] if 'selected_customers' in dir() else None

if _test_customer:
    _loader    = DataLoader(explainability_logger)
    _case_data = _loader.create_case_from_data(
        _test_customer['customer'],
        _test_customer['accounts'],
        _test_customer['transactions']
    )
    _result = risk_agent.analyze_case(_case_data)
    print("last_usage:", risk_agent.last_usage)
    print("classification:", _result.classification)
else:
    print("⚠️ Run Step 2 (screen_high_risk_customers) first to populate selected_customers")

## 📊 Step 1: Data Loading and Preprocessing

Load the financial data and prepare it for analysis.

In [4]:
# Step 1: Data Loading and Preprocessing

def load_and_preprocess_data():
    """
    Load CSV data and prepare for analysis.
    Applies dtype coercion and NaN handling consistent with foundation_sar.load_csv_data.
    """
    print("📊 Loading Financial Data...")

    customers_df    = pd.read_csv("../data/customers.csv", dtype={'ssn_last_4': str})
    accounts_df     = pd.read_csv("../data/accounts.csv")
    transactions_df = pd.read_csv("../data/transactions.csv")

    # Handle NaN in optional fields
    transactions_df['counterparty'] = transactions_df['counterparty'].fillna('')
    transactions_df['location']     = transactions_df['location'].fillna('')
    customers_df['phone']           = customers_df['phone'].fillna('')

    customers_data    = customers_df.to_dict('records')
    accounts_data     = accounts_df.to_dict('records')
    transactions_data = transactions_df.to_dict('records')

    print(f"📈 Loaded: {len(customers_data)} customers, "
          f"{len(accounts_data)} accounts, {len(transactions_data)} transactions")
    return customers_data, accounts_data, transactions_data

# Load data
customers_data, accounts_data, transactions_data = load_and_preprocess_data()

📊 Loading Financial Data...
📈 Loaded: 150 customers, 178 accounts, 4268 transactions


In [ ]:
customers_data[:2]

## 🎯 Step 2: Customer Risk Screening

Implement intelligent customer screening to identify high-risk cases for detailed analysis.

In [5]:
# Customer Risk Screening

def screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=5):
    """
    Risk-based customer screening.

    Criteria:
    - High risk rating (Medium or High)
    - Large transaction volume (>$100K total)
    - High transaction frequency (>50 transactions)

    Customers matching 2+ criteria are selected; results sorted by
    number of risk flags then total amount, returning top N.
    """
    selected_customers = []

    for customer in customers_data:
        customer_accounts = [
            acc for acc in accounts_data
            if acc['customer_id'] == customer['customer_id']
        ]
        customer_transactions = [
            txn for txn in transactions_data
            if any(txn['account_id'] == acc['account_id'] for acc in customer_accounts)
        ]

        total_amount      = sum(abs(txn['amount']) for txn in customer_transactions)
        transaction_count = len(customer_transactions)
        risk_rating       = customer['risk_rating']

        risk_flags = []
        if risk_rating in ['Medium', 'High']:
            risk_flags.append('high_risk_rating')
        if total_amount > 100000:
            risk_flags.append('large_amounts')
        if transaction_count > 50:
            risk_flags.append('high_frequency')

        if len(risk_flags) >= 2:
            selected_customers.append({
                'customer':          customer,
                'accounts':          customer_accounts,
                'transactions':      customer_transactions,
                'total_amount':      total_amount,
                'transaction_count': transaction_count,
                'risk_flags':        risk_flags
            })

    # Sort and slice OUTSIDE the loop
    selected_customers.sort(
        key=lambda x: (len(x['risk_flags']), x['total_amount']),
        reverse=True
    )
    result = selected_customers[:top_n]

    print(f"🔍 Screened {len(result)} high-risk customers from {len(customers_data)} total:")
    for c in result:
        print(f"   • {c['customer']['name']} | flags: {c['risk_flags']} | "
              f"txns: {c['transaction_count']} | total: ${c['total_amount']:,.0f}")
    return result

# Run customer screening
selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data)

🔍 Screened 5 high-risk customers from 150 total:
   • Jacqueline Rodriguez | flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | txns: 121 | total: $386,835
   • Michael Stanley | flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | txns: 90 | total: $380,634
   • Cindy Clayton | flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | txns: 101 | total: $286,343
   • Tracy Lewis | flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | txns: 94 | total: $274,025
   • Patrick Williams | flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | txns: 79 | total: $222,070


## 🤖 Step 3: Two-Stage AI Analysis with Human Gates

Implement the core two-stage workflow:
1. **Stage 1**: Risk Analyst performs Chain-of-Thought analysis
2. **Human Gate**: Review and decision to proceed
3. **Stage 2**: Compliance Officer generates ReACT narratives (only if approved)

In [ ]:
# Two-Stage AI Workflow with Human Decision Gates

def run_two_stage_sar_workflow(selected_customers):
    """
    Complete two-stage SAR processing workflow.

    Returns: (processed_cases, approved_sars, rejected_cases, audit_decisions, workflow_elapsed_s)
    audit_decisions entries include actual token counts from the LLM responses.
    """
    print("🤖 Two-Stage SAR Processing Workflow")

    processed_cases  = []
    approved_sars    = []
    rejected_cases   = []
    audit_decisions  = []
    workflow_start   = datetime.now()

    for i, customer_data in enumerate(selected_customers, 1):
        print(f"\n🔍 CUSTOMER {i}/{len(selected_customers)}: {customer_data['customer']['name']}")
        print("=" * 60)

        try:
            loader    = DataLoader(explainability_logger)
            case_data = loader.create_case_from_data(
                customer_data['customer'],
                customer_data['accounts'],
                customer_data['transactions']
            )

            # STAGE 1: Risk Analysis — timed
            print("🔍 STAGE 1: Risk Analysis")
            stage1_start  = datetime.now()
            risk_analysis = risk_agent.analyze_case(case_data)
            stage1_ms     = (datetime.now() - stage1_start).total_seconds() * 1000
            stage1_tokens = dict(risk_agent.last_usage) if risk_agent.last_usage else {}

            print(f"   Classification: {risk_analysis.classification}")
            print(f"   Confidence:     {risk_analysis.confidence_score:.2f}")
            print(f"   Risk Level:     {risk_analysis.risk_level}")
            print(f"   Reasoning:      {risk_analysis.reasoning[:120]}...")
            print(f"   Stage 1 time:   {stage1_ms/1000:.2f}s  |  tokens: {stage1_tokens.get('total_tokens', 'n/a')}")

            processed_cases.append({
                'case_id':        case_data.case_id,
                'customer_name':  case_data.customer.name,
                'classification': risk_analysis.classification,
                'risk_level':     risk_analysis.risk_level,
            })

            # HUMAN DECISION GATE
            decision       = input("\n🤔 Proceed with SAR filing? (yes/no): ").strip().lower()
            should_proceed = decision in ['yes', 'y']

            # Persist human decision to append-only audit log
            explainability_logger.log_human_decision(
                case_id=case_data.case_id,
                customer_name=case_data.customer.name,
                decision='PROCEED' if should_proceed else 'REJECT',
                ai_classification=risk_analysis.classification,
                ai_confidence=risk_analysis.confidence_score,
                ai_risk_level=risk_analysis.risk_level,
                rationale=f"Reviewer entered: '{decision}'"
            )

            stage2_ms     = None
            stage2_tokens = {}
            if should_proceed:
                # STAGE 2: Compliance Narrative — timed
                print("\n📝 STAGE 2: Compliance Narrative Generation")
                stage2_start      = datetime.now()
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
                stage2_ms         = (datetime.now() - stage2_start).total_seconds() * 1000
                stage2_tokens     = dict(compliance_agent.last_usage) if compliance_agent.last_usage else {}
                print(f"   Stage 2 time:   {stage2_ms/1000:.2f}s  |  tokens: {stage2_tokens.get('total_tokens', 'n/a')}")

                sar_document = create_sar_document(case_data, risk_analysis, compliance_review)
                save_sar_document(sar_document)
                approved_sars.append(sar_document)
                print(f"✅ SAR FILED: {sar_document['sar_metadata']['sar_id']}")
            else:
                rejected_cases.append({'case_id': case_data.case_id, 'reason': 'human_rejection'})
                print("❌ SAR REJECTED by human reviewer")

            audit_decisions.append({
                'case_id':           case_data.case_id,
                'customer_name':     case_data.customer.name,
                'decision':          'PROCEED' if should_proceed else 'REJECT',
                'ai_classification': risk_analysis.classification,
                'ai_confidence':     risk_analysis.confidence_score,
                'reviewer_decision': decision,
                'stage1_time_ms':    round(stage1_ms, 1),
                'stage2_time_ms':    round(stage2_ms, 1) if stage2_ms is not None else None,
                'stage1_tokens':     stage1_tokens,
                'stage2_tokens':     stage2_tokens,
            })

        except Exception as e:
            print(f"❌ Error processing {customer_data['customer']['name']}: {e}")
            import traceback; traceback.print_exc()

    workflow_elapsed_s = (datetime.now() - workflow_start).total_seconds()
    return processed_cases, approved_sars, rejected_cases, audit_decisions, workflow_elapsed_s

print("✅ run_two_stage_sar_workflow defined — run Step 4 cell first, then execute below.")

## 📄 Step 4: SAR Document Generation

Create complete, FinCEN-ready SAR documents with all required metadata.

In [7]:
# Students: Create complete SAR documents for regulatory submission

import datetime as _dt

class _DateEncoder(json.JSONEncoder):
    """JSON encoder that converts date/datetime objects to ISO strings."""
    def default(self, obj):
        if isinstance(obj, (_dt.date, _dt.datetime)):
            return obj.isoformat()
        return super().default(obj)


def create_sar_document(case_data, risk_analysis, compliance_review):
    """
    Create complete SAR document
    
    SAR document should include:
    1. SAR metadata (ID, filing date, type, checksum)
    2. Subject information (customer details)
    3. Suspicious activity description
    4. AI analysis results
    5. Compliance narrative
    6. Regulatory citations
    7. Filing institution information
    """
    print("📄 Creating SAR Document")
    print("📋 Generating unique SAR ID")
    print("📋 Including all required metadata")
    print("📋 Formatting for FinCEN submission")
    
    sar_id = f"SAR_{uuid.uuid4()}"
    filing_date = datetime.now().isoformat()

    # Date range of suspicious activity
    txn_dates = [txn.transaction_date for txn in case_data.transactions]
    activity_start = min(txn_dates).isoformat()
    activity_end   = max(txn_dates).isoformat()
    total_amount   = sum(abs(txn.amount) for txn in case_data.transactions)

    sar_document = {
        'sar_metadata': {
            'sar_id': sar_id,
            'filing_date': filing_date,
            'filing_type': 'Suspicious Activity Report',
            'ai_generated': True,
            'review_status': 'human_approved'
        },
        'filing_institution': {
            'name': 'First National Synthetic Bank',
            'address': '100 Compliance Ave, Washington, DC 20001',
            'ein': '12-3456789',
            'contact_name': 'SAR Processing Team',
            'contact_phone': '202-555-0100'
        },
        'subject_information': {
            'customer_name': case_data.customer.name,
            'customer_id': case_data.customer.customer_id,
            'address': case_data.customer.address,
            'date_of_birth': case_data.customer.date_of_birth.isoformat(),
            'occupation': case_data.customer.occupation,
            'phone': case_data.customer.phone,
            'customer_since': case_data.customer.customer_since.isoformat(),
            'risk_rating': case_data.customer.risk_rating
        },
        'account_information': [
            {
                'account_id': acc.account_id,
                'account_type': acc.account_type,
                'status': acc.status,
                'opening_date': acc.opening_date.isoformat(),
                'current_balance': acc.current_balance
            }
            for acc in case_data.accounts
        ],
        'suspicious_activity': {
            'classification': risk_analysis.classification,
            'risk_level': risk_analysis.risk_level,
            'confidence_score': risk_analysis.confidence_score,
            'activity_start_date': activity_start,
            'activity_end_date': activity_end,
            'total_amount_involved': round(total_amount, 2),
            'transaction_count': len(case_data.transactions),
            'narrative': compliance_review.narrative,
            'key_indicators': risk_analysis.key_indicators,
            'ai_reasoning': risk_analysis.reasoning
        },
        'regulatory_compliance': {
            'citations': getattr(compliance_review, 'regulatory_citations', []),
            'narrative_word_count': len(compliance_review.narrative.split()),
            'completeness_check': compliance_review.completeness_check,
            'compliance_status': 'approved'
        },
        'audit_trail': {
            'case_id': case_data.case_id,
            'processing_date': filing_date,
            'ai_agents_used': ['RiskAnalyst', 'ComplianceOfficer'],
            'human_reviewer': 'compliance_officer'
        }
    }
     
    return sar_document
    

def save_sar_document(sar_document):
    """Save SAR document to outputs directory"""
    print("📋 Saving SAR to ../outputs/filed_sars/ directory")
    os.makedirs("../outputs/filed_sars", exist_ok=True)
    filename = f"../outputs/filed_sars/{sar_document['sar_metadata']['sar_id']}.json"
    with open(filename, 'w') as f:
        json.dump(sar_document, f, indent=2, cls=_DateEncoder)

print("📄 SAR document generation functions defined")

📄 SAR document generation functions defined


In [8]:
# Run the complete workflow (must be after Step 4 defines create_sar_document / save_sar_document)
processed_cases, approved_sars, rejected_cases, audit_decisions, workflow_elapsed_s = run_two_stage_sar_workflow(selected_customers)

🤖 Two-Stage SAR Processing Workflow

🔍 CUSTOMER 1/5: Jacqueline Rodriguez
🔍 STAGE 1: Risk Analysis
   Classification: Money_Laundering
   Confidence:     0.85
   Risk Level:     High
   Reasoning:      Step 1: High volume of transactions and high balances inconsistent with occupation. Step 2: Large wire transfers and mul...
   Stage 1 time:   4.84s
❌ SAR REJECTED by human reviewer

🔍 CUSTOMER 2/5: Michael Stanley
🔍 STAGE 1: Risk Analysis
   Classification: Money_Laundering
   Confidence:     0.85
   Risk Level:     High
   Reasoning:      Step 1: Reviewed customer's profile, accounts, and transactions. Step 2: Identified unusual volume and high-risk methods...
   Stage 1 time:   3.79s
❌ SAR REJECTED by human reviewer

🔍 CUSTOMER 3/5: Cindy Clayton
🔍 STAGE 1: Risk Analysis
   Classification: Money_Laundering
   Confidence:     0.85
   Risk Level:     High
   Reasoning:      Step 1: High volume of transactions inconsistent with income. Step 2: Multiple large ATM withdrawals and deposits.

## 📊 Step 5: Workflow Metrics and Analysis

Analyze the efficiency and effectiveness of your AI-powered SAR processing system.

In [9]:
# Step 5: Workflow Metrics and Analysis

# GPT-4 pricing per million tokens (adjust if using a different model)
INPUT_COST_PER_M  = 30.00   # $30 per million input tokens
OUTPUT_COST_PER_M = 60.00   # $60 per million output tokens

def _tokens_to_cost(prompt_tokens, completion_tokens):
    """Convert actual token counts to USD cost."""
    return (prompt_tokens / 1_000_000) * INPUT_COST_PER_M + \
           (completion_tokens / 1_000_000) * OUTPUT_COST_PER_M


def analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases,
                                audit_decisions, workflow_elapsed_s=None):
    """
    Calculate and save a durable performance metrics report using actual token usage.

    Covers:
    - Volume counts and rates
    - Total and per-stage AI processing time
    - Stage 2 calls avoided by the human gate
    - Actual cost from real prompt/completion token counts
    - Throughput (cases per minute of AI time)
    """
    print("📊 Workflow Efficiency Analysis")
    print("=" * 60)

    total_cases    = len(processed_cases)
    approved_count = len(approved_sars)
    rejected_count = len(rejected_cases)
    approval_rate  = approved_count / total_cases if total_cases > 0 else 0
    rejection_rate = rejected_count / total_cases if total_cases > 0 else 0

    # ── Per-stage timing ──────────────────────────────────────────
    stage1_times = [d['stage1_time_ms'] for d in audit_decisions if d.get('stage1_time_ms') is not None]
    stage2_times = [d['stage2_time_ms'] for d in audit_decisions if d.get('stage2_time_ms') is not None]

    total_stage1_s = sum(stage1_times) / 1000
    total_stage2_s = sum(stage2_times) / 1000
    total_ai_s     = total_stage1_s + total_stage2_s
    avg_stage1_s   = (sum(stage1_times) / len(stage1_times) / 1000) if stage1_times else 0
    avg_stage2_s   = (sum(stage2_times) / len(stage2_times) / 1000) if stage2_times else 0

    # ── Actual cost from real token usage ────────────────────────
    actual_cost = 0.0
    for d in audit_decisions:
        s1 = d.get('stage1_tokens', {})
        s2 = d.get('stage2_tokens', {})
        actual_cost += _tokens_to_cost(s1.get('prompt_tokens', 0), s1.get('completion_tokens', 0))
        actual_cost += _tokens_to_cost(s2.get('prompt_tokens', 0), s2.get('completion_tokens', 0))

    # Hypothetical cost if Stage 2 had run on rejected cases too
    avg_s2_cost = (actual_cost - sum(
        _tokens_to_cost(d.get('stage1_tokens', {}).get('prompt_tokens', 0),
                        d.get('stage1_tokens', {}).get('completion_tokens', 0))
        for d in audit_decisions
    )) / approved_count if approved_count else 0

    hypothetical_cost = actual_cost + rejected_count * avg_s2_cost
    cost_saved        = hypothetical_cost - actual_cost
    savings_pct       = (cost_saved / hypothetical_cost * 100) if hypothetical_cost > 0 else 0

    total_prompt     = sum(d.get('stage1_tokens', {}).get('prompt_tokens', 0) +
                           d.get('stage2_tokens', {}).get('prompt_tokens', 0)
                           for d in audit_decisions)
    total_completion = sum(d.get('stage1_tokens', {}).get('completion_tokens', 0) +
                           d.get('stage2_tokens', {}).get('completion_tokens', 0)
                           for d in audit_decisions)

    # ── Throughput ────────────────────────────────────────────────
    ai_minutes = total_ai_s / 60
    throughput = total_cases / ai_minutes if ai_minutes > 0 else 0

    # ── Print report ──────────────────────────────────────────────
    print(f"\n📈 VOLUME METRICS:")
    print(f"   Total Cases Processed : {total_cases}")
    print(f"   SARs Filed            : {approved_count}  ({approval_rate:.1%})")
    print(f"   Cases Rejected        : {rejected_count}  ({rejection_rate:.1%})")
    print(f"   Stage 2 Calls Avoided : {rejected_count} of {total_cases}")

    print(f"\n⏱  TIMING METRICS:")
    if workflow_elapsed_s is not None:
        print(f"   End-to-End Wall Time  : {workflow_elapsed_s:.1f}s  (includes human review)")
    print(f"   Total AI Time         : {total_ai_s:.1f}s")
    print(f"   ├─ Stage 1 (total)    : {total_stage1_s:.1f}s  |  avg {avg_stage1_s:.1f}s/case")
    print(f"   └─ Stage 2 (total)    : {total_stage2_s:.1f}s  |  avg {avg_stage2_s:.1f}s/case")

    print(f"\n🔢 TOKEN USAGE  (actual from API):")
    print(f"   Total Input Tokens    : {total_prompt:,}")
    print(f"   Total Output Tokens   : {total_completion:,}")
    print(f"   Total Tokens          : {total_prompt + total_completion:,}")

    print(f"\n💰 COST  (actual, ${INPUT_COST_PER_M}/M input · ${OUTPUT_COST_PER_M}/M output):")
    print(f"   Two-stage actual cost : ${actual_cost:.4f}")
    print(f"   Single-stage estimate : ${hypothetical_cost:.4f}  (if Stage 2 ran on all cases)")
    print(f"   Estimated savings     : ${cost_saved:.4f}  ({savings_pct:.1f}% reduction)")

    print(f"\n🚀 THROUGHPUT:")
    print(f"   AI Processing Rate    : {throughput:.1f} cases/minute")

    # ── Save durable report ───────────────────────────────────────
    os.makedirs("../outputs", exist_ok=True)
    report = {
        "report_timestamp": datetime.now().isoformat(),
        "volume": {
            "total_cases": total_cases,
            "approved": approved_count,
            "rejected": rejected_count,
            "approval_rate": round(approval_rate, 4),
            "rejection_rate": round(rejection_rate, 4),
            "stage2_calls_avoided": rejected_count,
        },
        "timing_seconds": {
            "end_to_end_wall_time": round(workflow_elapsed_s, 2) if workflow_elapsed_s else None,
            "total_ai_time":        round(total_ai_s, 2),
            "total_stage1":         round(total_stage1_s, 2),
            "total_stage2":         round(total_stage2_s, 2),
            "avg_stage1_per_case":  round(avg_stage1_s, 2),
            "avg_stage2_per_case":  round(avg_stage2_s, 2),
        },
        "token_usage": {
            "total_prompt_tokens":     total_prompt,
            "total_completion_tokens": total_completion,
            "total_tokens":            total_prompt + total_completion,
        },
        "cost_usd": {
            "pricing_model":         f"${INPUT_COST_PER_M}/M input, ${OUTPUT_COST_PER_M}/M output",
            "two_stage_actual":      round(actual_cost, 6),
            "single_stage_estimate": round(hypothetical_cost, 6),
            "estimated_savings":     round(cost_saved, 6),
            "savings_pct":           round(savings_pct, 2),
        },
        "throughput": {
            "cases_per_minute_ai_time": round(throughput, 2),
        },
        "per_case_detail": [
            {
                "case_id":        d["case_id"],
                "customer_name":  d["customer_name"],
                "decision":       d["decision"],
                "stage1_time_ms": d.get("stage1_time_ms"),
                "stage2_time_ms": d.get("stage2_time_ms"),
                "stage1_tokens":  d.get("stage1_tokens", {}),
                "stage2_tokens":  d.get("stage2_tokens", {}),
            }
            for d in audit_decisions
        ],
    }

    report_path = "../outputs/workflow_metrics_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n📁 Full report saved to: {report_path}")

    return report


def validate_ai_decisions(audit_decisions):
    """Analyse AI decision patterns from the audit trail."""
    if not audit_decisions:
        print("⚠️  No audit decisions to analyse.")
        return

    print("\n🔍 AI DECISION ANALYSIS")
    print("=" * 50)

    from collections import Counter
    classifications = Counter(d['ai_classification'] for d in audit_decisions)
    print("\n📊 Classification Breakdown:")
    for cls, count in classifications.most_common():
        print(f"   {cls:<20} {count} case{'s' if count != 1 else ''}")

    scores = [d['ai_confidence'] for d in audit_decisions]
    print(f"\n📉 Confidence Scores:")
    print(f"   Min:     {min(scores):.2f}")
    print(f"   Max:     {max(scores):.2f}")
    print(f"   Average: {sum(scores)/len(scores):.2f}")

    high_conf   = [s for s in scores if s >= 0.8]
    medium_conf = [s for s in scores if 0.5 <= s < 0.8]
    low_conf    = [s for s in scores if s < 0.5]
    print(f"   High (≥0.8):      {len(high_conf)} cases")
    print(f"   Medium (0.5–0.8): {len(medium_conf)} cases")
    print(f"   Low (<0.5):       {len(low_conf)} cases")

    proceeded = [d for d in audit_decisions if d['decision'] == 'PROCEED']
    rejected  = [d for d in audit_decisions if d['decision'] == 'REJECT']
    print(f"\n👤 Human Decision Patterns:")
    print(f"   Approved: {len(proceeded)} | Rejected: {len(rejected)}")

    if rejected:
        avg_conf_rejected = sum(d['ai_confidence'] for d in rejected) / len(rejected)
        print(f"   Avg confidence on rejected cases: {avg_conf_rejected:.2f}")
        rejected_cls = Counter(d['ai_classification'] for d in rejected)
        print(f"   Classifications rejected by reviewer:")
        for cls, count in rejected_cls.most_common():
            print(f"     • {cls}: {count}")

    print(f"\n📋 Case-by-Case Summary:")
    for d in audit_decisions:
        icon = "✅" if d['decision'] == 'PROCEED' else "❌"
        s1 = f"{d['stage1_time_ms']/1000:.1f}s" if d.get('stage1_time_ms') else "n/a"
        s2 = f"{d['stage2_time_ms']/1000:.1f}s" if d.get('stage2_time_ms') else "skipped"
        t1 = d.get('stage1_tokens', {}).get('total_tokens', 'n/a')
        t2 = d.get('stage2_tokens', {}).get('total_tokens', 'n/a') if d.get('stage2_tokens') else 'skipped'
        print(f"   {icon} {d['customer_name']:<25} {d['ai_classification']:<20} "
              f"conf:{d['ai_confidence']:.2f}  s1:{s1}({t1}tok)  s2:{s2}({t2}tok)  → {d['decision']}")


# Run analysis
analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions, workflow_elapsed_s)
validate_ai_decisions(audit_decisions)

📊 Workflow Efficiency Analysis

📈 VOLUME METRICS:
   Total Cases Processed : 5
   SARs Filed            : 3  (60.0%)
   Cases Rejected        : 2  (40.0%)
   Stage 2 Calls Avoided : 2 of 5

⏱  TIMING METRICS:
   End-to-End Wall Time  : 141.6s  (includes human review)
   Total AI Time         : 120.1s
   ├─ Stage 1 (total)    : 63.8s  |  avg 12.8s/case
   └─ Stage 2 (total)    : 56.3s  |  avg 18.8s/case

🔢 TOKEN USAGE  (actual from API):
   Total Input Tokens    : 0
   Total Output Tokens   : 0
   Total Tokens          : 0

💰 COST  (actual, $30.0/M input · $60.0/M output):
   Two-stage actual cost : $0.0000
   Single-stage estimate : $0.0000  (if Stage 2 ran on all cases)
   Estimated savings     : $0.0000  (0.0% reduction)

🚀 THROUGHPUT:
   AI Processing Rate    : 2.5 cases/minute

📁 Full report saved to: ../outputs/workflow_metrics_report.json

🔍 AI DECISION ANALYSIS

📊 Classification Breakdown:
   Money_Laundering     3 cases
   Structuring          2 cases

📉 Confidence Scores:
   M

## 🏁 Step 6: Complete System Demonstration

Test your complete system with comprehensive scenarios to validate production readiness.

In [ ]:
# Complete System Demonstration

def demonstrate_complete_system():
    """
    Run complete system demonstration.
    
    1. Load fresh data from CSVs
    2. Screen top 3 high-risk customers
    3. Run full two-stage workflow with human gates
    4. Print efficiency metrics and save report
    5. Report output locations
    """
    print("🚀 Running complete system test...")

    customers_data, accounts_data, transactions_data = load_and_preprocess_data()

    selected_customers = screen_high_risk_customers(
        customers_data, accounts_data, transactions_data, top_n=3
    )

    processed_cases, approved_sars, rejected_cases, audit_decisions, workflow_elapsed_s = \
        run_two_stage_sar_workflow(selected_customers)

    analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases,
                                audit_decisions, workflow_elapsed_s)

    print(f"\n🎉 System demonstration complete!")
    print(f"📄 SAR documents saved to:    ../outputs/filed_sars/")
    print(f"📊 Audit logs saved to:       ../outputs/audit_logs/")
    print(f"📈 Metrics report saved to:   ../outputs/workflow_metrics_report.json")

demonstrate_complete_system()

## 📝 Implementation Checklist

### ✅ Workflow Integration Deliverables
- [ ] **Data Loading**: Load and preprocess CSV data with proper error handling
- [ ] **Customer Screening**: Implement risk-based screening to identify high-risk cases
- [ ] **Two-Stage Workflow**: Build complete Risk Analyst → Human Gate → Compliance Officer flow
- [ ] **Human Decision Gates**: Implement interactive approval/rejection points
- [ ] **SAR Document Generation**: Create complete FinCEN-ready documents with metadata
- [ ] **Audit Trail Creation**: Log all decisions and reasoning for regulatory examination
- [ ] **Efficiency Metrics**: Calculate cost optimization and processing efficiency
- [ ] **System Demonstration**: Test complete workflow with multiple scenarios

### ✅ Testing and Validation Requirements
- [ ] **Component Validation**: Verify all foundation components and agents are available
- [ ] **Integration Testing**: Run comprehensive test suites for all components with proper sys.path setup
- [ ] **End-to-End Testing**: Test complete workflow with automated scenarios
- [ ] **Error Handling Testing**: Validate graceful handling of edge cases and failures
- [ ] **Output Validation**: Ensure SAR documents meet regulatory standards
- [ ] **Performance Testing**: Measure workflow efficiency and processing times

### ✅ Technical Requirements
- [ ] **Error Handling**: Robust exception handling for all workflow steps
- [ ] **Data Validation**: Proper validation of all inputs and outputs
- [ ] **File Management**: Organize outputs in appropriate directories
- [ ] **Logging**: Comprehensive audit logging for compliance
- [ ] **Performance**: Efficient processing of multiple cases
- [ ] **User Experience**: Clear prompts and feedback for human reviewers
- [ ] **Test Infrastructure**: Proper test imports and sys.path configuration

### ✅ Business Requirements  
- [ ] **Regulatory Compliance**: Ensure all SAR documents meet FinCEN requirements
- [ ] **Cost Optimization**: Demonstrate savings from two-stage processing
- [ ] **Audit Readiness**: Create examination-ready documentation
- [ ] **Quality Assurance**: Validate AI decisions with human oversight
- [ ] **Scalability**: Design for processing larger datasets
- [ ] **Production Readiness**: Complete testing validates system reliability

## 🎯 Success Criteria

By completion, your integrated system should:
- ✅ Process real financial data with proper validation
- ✅ Execute complete two-stage AI workflow with human gates
- ✅ Generate regulatory-compliant SAR documents
- ✅ Create comprehensive audit trails for all decisions
- ✅ Demonstrate measurable cost optimization benefits
- ✅ Handle errors gracefully and provide clear user feedback
- ✅ Pass all integration and end-to-end tests
- ✅ Meet production-ready quality standards

## 🚀 Next Steps

1. **Complete Implementation**: Fill in all TODO sections with working code
2. **Run Integration Tests**: Validate all components work together properly
3. **Execute End-to-End Tests**: Test complete workflow with automated scenarios
4. **Test Thoroughly**: Run complete workflow with various manual scenarios
5. **Validate Outputs**: Ensure SAR documents meet regulatory requirements
6. **Document Results**: Create final project documentation and metrics
7. **Prepare Presentation**: Demonstrate your system's capabilities and business value

**Congratulations on building a complete AI-powered SAR processing system! 🎉**

## 🧪 Step 7: Workflow Testing and Validation

Before finalizing your implementation, validate your complete system with comprehensive testing.

In [ ]:
# 🧪 Workflow Integration Testing
# Validate your complete system with integration tests

import sys
import os

# Add tests directory to Python path for importing test modules
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Added tests directory to Python path: {tests_path}")

def run_integration_tests():
    """
    Run comprehensive integration tests to validate the complete workflow
    """
    print("🧪 Comprehensive Integration Testing")
    
    try:
        import pytest

        print("🔍 Running Foundation Component Tests...")
        foundation_result = pytest.main([
            f"{tests_path}/test_foundation.py",
            "-v",
            "--tb=short",
            "-q"
        ])

        print("🔍 Running Risk Analyst Agent Tests...")
        risk_result = pytest.main([
            f"{tests_path}/test_risk_analyst.py",
            "-v",
            "--tb=short",
            "-q"
        ])

        print("📝 Running Compliance Officer Agent Tests...")
        compliance_result = pytest.main([
            f"{tests_path}/test_compliance_officer.py",
            "-v",
            "--tb=short",
            "-q"
        ])

        all_passed = foundation_result == 0 and risk_result == 0 and compliance_result == 0

        print("\n" + "="*60)
        print("📊 INTEGRATION TEST RESULTS:")
        print(f"   Foundation Components: {'✅ PASS' if foundation_result == 0 else '❌ FAIL'}")
        print(f"   Risk Analyst Agent:    {'✅ PASS' if risk_result == 0 else '❌ FAIL'}")
        print(f"   Compliance Officer:    {'✅ PASS' if compliance_result == 0 else '❌ FAIL'}")
        print(f"   Overall Status:        {'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED'}")

        if all_passed:
            print("\n🎉 Your system is ready for production workflow testing!")
        else:
            print("\n⚠️ Fix failing tests before running the complete workflow.")

        return all_passed

    except ImportError as e:
        print(f"❌ Import Error: {e}")
        return False

def validate_workflow_components():
    """Validate that all required components are available for integration"""
    print("🔍 Validating Workflow Components")

    components_status = {
        'foundation_sar': False,
        'risk_analyst_agent': False,
        'compliance_officer_agent': False,
        'test_modules': False
    }

    try:
        from foundation_sar import CustomerData, CaseData, ExplainabilityLogger, DataLoader
        components_status['foundation_sar'] = True
        print("✅ Foundation components available")
    except ImportError:
        print("❌ Foundation components not available")

    try:
        from risk_analyst_agent import RiskAnalystAgent
        components_status['risk_analyst_agent'] = True
        print("✅ Risk Analyst Agent available")
    except ImportError:
        print("❌ Risk Analyst Agent not available")

    try:
        from compliance_officer_agent import ComplianceOfficerAgent
        components_status['compliance_officer_agent'] = True
        print("✅ Compliance Officer Agent available")
    except ImportError:
        print("❌ Compliance Officer Agent not available")

    try:
        from test_foundation import TestCustomerData
        from test_risk_analyst import TestRiskAnalystAgent
        from test_compliance_officer import TestComplianceOfficerAgent
        components_status['test_modules'] = True
        print("✅ Test modules available")
    except ImportError:
        print("❌ Test modules not available")

    all_ready = all(components_status.values())

    print(f"\n📊 Component Status: {'✅ ALL READY' if all_ready else '⚠️ INCOMPLETE'}")
    if not all_ready:
        print("💡 Complete missing components before running integration tests")

    return all_ready

# Run component validation
components_ready = validate_workflow_components()

# Run integration tests if components are ready
if components_ready:
    print("\n🚀 All components ready - running integration tests!")
    run_integration_tests()
else:
    print("\n📋 Complete component implementation first, then run integration tests")

In [ ]:
# 🎯 End-to-End Workflow Testing
# Test the complete workflow with known test scenarios (automated decisions)

def test_complete_workflow():
    """
    Test the complete SAR processing workflow end-to-end with automated decisions.
    Uses simulated human approvals so the test runs without interactive input.
    """
    print("🎯 End-to-End Workflow Testing")

    try:
        print("🚀 Starting end-to-end workflow test...")

        # Test data preparation
        print("📊 Loading test data...")
        customers_data, accounts_data, transactions_data = load_and_preprocess_data()

        if not customers_data:
            print("⚠️ No test data available - implement data loading first")
            return False

        # Test customer screening
        print("🔍 Testing customer screening...")
        selected_customers = screen_high_risk_customers(
            customers_data, accounts_data, transactions_data, top_n=2
        )

        if not selected_customers:
            print("⚠️ No customers selected - check screening criteria")
            return False

        print(f"✅ Selected {len(selected_customers)} customers for testing")

        # Test workflow with automated decisions (simulate approval for all cases)
        print("🤖 Testing automated workflow (simulating human approval)...")
        test_results = {
            'cases_processed': 0,
            'sars_generated': 0,
            'errors': []
        }

        for customer_data in selected_customers:
            try:
                # Create case
                loader = DataLoader(explainability_logger)
                case_data = loader.create_case_from_data(
                    customer_data['customer'],
                    customer_data['accounts'],
                    customer_data['transactions']
                )

                # Test Risk Analyst
                risk_analysis = risk_agent.analyze_case(case_data)
                assert hasattr(risk_analysis, 'classification'), "Risk analysis missing classification"
                assert hasattr(risk_analysis, 'confidence_score'), "Risk analysis missing confidence"

                # Test Compliance Officer (simulate approval)
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
                assert hasattr(compliance_review, 'narrative'), "Compliance review missing narrative"

                # Test SAR generation
                sar_document = create_sar_document(case_data, risk_analysis, compliance_review)
                assert sar_document, "SAR document generation failed"

                # Save SAR
                save_sar_document(sar_document)

                test_results['cases_processed'] += 1
                test_results['sars_generated'] += 1

                print(f"✅ Successfully processed: {customer_data['customer']['name']}")
                print(f"   Classification: {risk_analysis.classification} | Confidence: {risk_analysis.confidence_score:.2f}")
                print(f"   Narrative ({len(compliance_review.narrative.split())} words): {compliance_review.narrative[:80]}...")

            except Exception as e:
                test_results['errors'].append(f"Error processing {customer_data['customer']['name']}: {e}")
                print(f"❌ Error: {customer_data['customer']['name']}: {e}")

        # Test results summary
        print("\n" + "="*60)
        print("📊 END-TO-END TEST RESULTS:")
        print(f"   Cases Processed: {test_results['cases_processed']}")
        print(f"   SARs Generated:  {test_results['sars_generated']}")
        print(f"   Errors:          {len(test_results['errors'])}")

        if test_results['errors']:
            print("❌ Test Errors:")
            for error in test_results['errors']:
                print(f"     • {error}")

        success = len(test_results['errors']) == 0 and test_results['cases_processed'] > 0

        if success:
            print("\n🎉 END-TO-END TEST PASSED!")
            print("✅ Your complete workflow is ready for production use!")
        else:
            print("\n⚠️ END-TO-END TEST HAD ISSUES")
            print("📝 Fix the errors above before deploying to production")

        return success

    except Exception as e:
        print(f"❌ End-to-end test failed: {e}")
        import traceback
        traceback.print_exc()
        return False

# Run end-to-end test
test_success = test_complete_workflow()